In [ ]:
!pip install -q langchain langchain-community langchain-chroma langgraph langchain-google-genai pypdf chromadb langgraph python-dotenv google-generativeai

## Configuración y API Key

In [ ]:
import os
from google.colab import userdata

GEMINI_API_KEY = userdata.get("Insertar el nombre de API Key Gemini")
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

# -- Parámetros globales del proyecto --------------------------
TEMA_EXPERTO    = "Cine y películas de Quentin Tarantino"
PDF_DIR         = "/content/pdfs"
CHROMA_DIR      = "/content/chroma_db"
COLLECTION_NAME = "tarantino_expert"
CHUNK_SIZE      = 800
CHUNK_OVERLAP   = 100
TOP_K           = 5
EMBEDDING_MODEL = "models/embedding-001"
LLM_MODEL       = "gemini-2.5-flash-lite"


## Subida de documentación

In [ ]:
import os, shutil
from google.colab import files

os.makedirs(PDF_DIR, exist_ok=True)
uploaded = files.upload()

for filename in uploaded.keys():
    dest = os.path.join(PDF_DIR, filename)
    shutil.move(filename, dest)

pdf_files = [f for f in os.listdir(PDF_DIR) if f.lower().endswith(".pdf")]
print(f"\n📚 Total PDFs disponibles: {len(pdf_files)}")
for f in pdf_files:
    kb = os.path.getsize(os.path.join(PDF_DIR, f)) / 1024
    print(f"   • {f}  ({kb:.1f} KB)")

Saving Death_Proof.pdf to Death_Proof.pdf
Saving Django_Unchained.pdf to Django_Unchained.pdf
Saving Inglourious_Basterds.pdf to Inglourious_Basterds.pdf
Saving Jackie_Brown.pdf to Jackie_Brown.pdf
Saving Kill_Bill__Volumen_1.pdf to Kill_Bill__Volumen_1.pdf
Saving Kill_Bill__Volumen_2.pdf to Kill_Bill__Volumen_2.pdf
Saving Once_Upon_a_Time_in_Hollywood.pdf to Once_Upon_a_Time_in_Hollywood.pdf
Saving Pulp_Fiction.pdf to Pulp_Fiction.pdf
Saving Reservoir_Dogs.pdf to Reservoir_Dogs.pdf
Saving The_Hateful_Eight.pdf to The_Hateful_Eight.pdf

📚 Total PDFs disponibles: 10
   • Inglourious_Basterds.pdf  (410.4 KB)
   • Reservoir_Dogs.pdf  (283.6 KB)
   • The_Hateful_Eight.pdf  (216.6 KB)
   • Django_Unchained.pdf  (247.3 KB)
   • Pulp_Fiction.pdf  (568.3 KB)
   • Jackie_Brown.pdf  (211.8 KB)
   • Death_Proof.pdf  (238.0 KB)
   • Kill_Bill__Volumen_1.pdf  (267.9 KB)
   • Kill_Bill__Volumen_2.pdf  (279.0 KB)
   • Once_Upon_a_Time_in_Hollywood.pdf  (784.0 KB)


## Limpieza y chunking



In [ ]:
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def clean_text(text: str) -> str:
    """Limpieza exhaustiva para PDFs de Wikipedia."""

    # Eliminar URLs completas
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)

    # Eliminar referencias numericas tipo [1], [cita requerida], [2 3]
    text = re.sub(r'\[\d+\]', '', text)
    text = re.sub(r'\[cita requerida\]', '', text)
    text = re.sub(r'\[nota \d+\]', '', text)

    # Eliminar notas al pie (lineas que empiezan por numero + punto)
    text = re.sub(r'^\d{1,3}\.\s+.{0,20}(Consultado|Archivado|ISBN|doi:).*$',
                  '', text, flags=re.MULTILINE)

    # Eliminar lineas de solo numeros o codigos cortos
    text = re.sub(r'^\s*[\d\.\-\+]+\s*$', '', text, flags=re.MULTILINE)

    # Eliminar patrones de tabla fragmentada (lineas muy cortas sueltas)
    lines = text.split('\n')
    lines = [l for l in lines if len(l.strip()) > 15 or l.strip() == '']
    text = '\n'.join(lines)

    # Normalizar espacios y saltos
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'\.{4,}', '...', text)

    return text.strip()


def load_pdfs(folder: str):
    """Carga todos los PDFs de una carpeta página a página."""
    docs = []
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(".pdf"):
            continue
        loader = PyPDFLoader(os.path.join(folder, fname))
        pages  = loader.load()
        docs.extend(pages)
        print(f" {fname} → {len(pages)} páginas")
    return docs


raw_docs = load_pdfs(PDF_DIR)

# Limpiar contenido y filtrar páginas vacías
for doc in raw_docs:
    doc.page_content = clean_text(doc.page_content)
raw_docs = [d for d in raw_docs if len(d.page_content.strip()) > 50]

# Chunking con ventana deslizante
splitter = RecursiveCharacterTextSplitter(
    chunk_size      = CHUNK_SIZE,
    chunk_overlap   = CHUNK_OVERLAP,
    separators      = ["\n\n", "\n", ". ", " ", ""],
    length_function = len,
)
chunks = splitter.split_documents(raw_docs)

print(f"\n📊 Resultados del chunking:")
print(f"   • Páginas válidas  : {len(raw_docs)}")
print(f"   • Total chunks     : {len(chunks)}")

 Death_Proof.pdf → 5 páginas
 Django_Unchained.pdf → 6 páginas
 Inglourious_Basterds.pdf → 9 páginas
 Jackie_Brown.pdf → 5 páginas
 Kill_Bill__Volumen_1.pdf → 8 páginas
 Kill_Bill__Volumen_2.pdf → 7 páginas
 Once_Upon_a_Time_in_Hollywood.pdf → 23 páginas
 Pulp_Fiction.pdf → 17 páginas
 Reservoir_Dogs.pdf → 7 páginas
 The_Hateful_Eight.pdf → 5 páginas

📊 Resultados del chunking:
   • Páginas válidas  : 90
   • Total chunks     : 360


## Carga de docoumentos en ChromaDB con Gemini Embeddings

In [ ]:
import time
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

# Creo la carpeta donde se guarda la base de datos
os.makedirs(CHROMA_DIR, exist_ok=True)

# Cuantos chunks mando a la vez y cuanto espero entre lotes
BATCH_SIZE   = 20
WAIT_SECONDS = 15

# Modelo de embeddings de Gemini
embeddings = GoogleGenerativeAIEmbeddings(
    model     = "gemini-embedding-001",
    task_type = "retrieval_document",
)

# Divido los chunks en grupos pequeños
lotes = [chunks[i:i+BATCH_SIZE] for i in range(0, len(chunks), BATCH_SIZE)]
vectorstore = None

print("Indexando documentos en ChromaDB...")

# Recorro cada lote y lo indexo
for numero, lote in enumerate(lotes):
    print(f"Lote {numero+1} de {len(lotes)}...", end=" ")

    if vectorstore is None:
        # El primer lote crea la base de datos
        vectorstore = Chroma.from_documents(
            documents         = lote,
            embedding         = embeddings,
            collection_name   = COLLECTION_NAME,
            persist_directory = CHROMA_DIR,
        )
    else:
        # Los siguientes se añaden a la base de datos existente
        vectorstore.add_documents(lote)

    print(f"OK ({vectorstore._collection.count()} chunks)")

    # Espero entre lotes para no superar el limite de la API
    if numero < len(lotes) - 1:
        time.sleep(WAIT_SECONDS)

# Guardo en disco
vectorstore.persist()
print(f"Indexacion completada: {vectorstore._collection.count()} chunks guardados")

Indexando documentos en ChromaDB...
Lote 1 de 18... OK (580 chunks)
Lote 2 de 18... OK (600 chunks)
Lote 3 de 18... OK (620 chunks)
Lote 4 de 18... OK (640 chunks)
Lote 5 de 18... OK (660 chunks)
Lote 6 de 18... OK (680 chunks)
Lote 7 de 18... OK (700 chunks)
Lote 8 de 18... OK (720 chunks)
Lote 9 de 18... OK (740 chunks)
Lote 10 de 18... OK (760 chunks)
Lote 11 de 18... OK (780 chunks)
Lote 12 de 18... OK (800 chunks)
Lote 13 de 18... OK (820 chunks)
Lote 14 de 18... OK (840 chunks)
Lote 15 de 18... OK (860 chunks)
Lote 16 de 18... OK (880 chunks)
Lote 17 de 18... OK (900 chunks)
Lote 18 de 18... OK (920 chunks)
Indexacion completada: 920 chunks guardados


/tmp/ipykernel_31502/2879574210.py:47: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


## Preparación del system prompt

In [ ]:
SYSTEM_PROMPT = """
Eres un asistente experto en el cine de Quentin Tarantino.

Tu conocimiento proviene EXCLUSIVAMENTE de los fragmentos de documentos que se te
proporcionan en cada consulta. Sigue estas reglas sin excepción:

1. Basa tus respuestas únicamente en los fragmentos recuperados.
   No uses conocimiento externo ni inventes información.

2. Si los fragmentos no contienen información suficiente, dilo:
   "No tengo esa información en mi base de conocimiento."

3. MEMORIA: Tienes acceso al historial de conversación. Mantén coherencia con
   respuestas anteriores y haz referencia a ellas cuando sea relevante.

4. IDIOMA: Responde siempre en español.

5. FORMATO: Usa listas y negritas cuando mejore la legibilidad.
""".strip()

## Agente LangGraph

In [ ]:
from typing import TypedDict, Annotated, List
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# Modelo de lenguaje
llm = ChatGoogleGenerativeAI(
    model = LLM_MODEL,
    google_api_key = GEMINI_API_KEY,
    temperature = 0.2,
    convert_system_message_to_human = True,
)

# Buscador en ChromaDB
retriever = vectorstore.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k": TOP_K, "fetch_k": TOP_K * 3},
)

# Estado compartido entre nodos
class AgentState(TypedDict):
    messages : Annotated[List[BaseMessage], add_messages]
    context  : str
    query    : str

# Nodo 1: busca documentos relevantes en ChromaDB
def retrieve_node(state):
    docs = retriever.invoke(state["query"])
    context = "\n\n".join([doc.page_content for doc in docs])
    return {**state, "context": context}

# Nodo 2: genera la respuesta con Gemini
def generate_node(state):
    mensajes = [
        SystemMessage(content=SYSTEM_PROMPT),
        *state["messages"][:-1],
        HumanMessage(content=f"Contexto:\n{state['context']}\n\nPregunta: {state['query']}"),
    ]
    respuesta = llm.invoke(mensajes)
    return {**state, "messages": [AIMessage(content=respuesta.content)]}

# Construccion del grafo
grafo = StateGraph(AgentState)
grafo.add_node("retrieve", retrieve_node)
grafo.add_node("generate", generate_node)
grafo.set_entry_point("retrieve")
grafo.add_edge("retrieve", "generate")
grafo.add_edge("generate", END)

agent = grafo.compile()
print("Agente compilado correctamente")


/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Agente compilado correctamente


## Agregar memoria de conversación

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# Clase para guardar el historial de conversacion
class ConversationMemory:

    def __init__(self, max_turns=10):
        self.history = []
        self.max_turns = max_turns

    def add_turn(self, pregunta, respuesta):
        self.history.append(HumanMessage(content=pregunta))
        self.history.append(AIMessage(content=respuesta))
        # Si el historial es muy largo lo recorto
        if len(self.history) > self.max_turns * 2:
            self.history = self.history[-(self.max_turns * 2):]

    def get(self):
        return list(self.history)

    def clear(self):
        self.history = []
        print("Historial borrado.")

    def show(self):
        if not self.history:
            print("Sin historial todavia")
            return
        for msg in self.history:
            rol = "Usuario" if isinstance(msg, HumanMessage) else "Agente"
            print(f"  {rol}: {msg.content[:100]}...")


# Funcion para hacer preguntas al agente
def ask(question, verbose=True):
    initial_state = {
        "messages" : memory.get() + [HumanMessage(content=question)],
        "context"  : "",
        "query"    : question,
    }

    result = agent.invoke(initial_state)
    answer = result["messages"][-1].content

    memory.add_turn(question, answer)

    if verbose:
        print(f"\nPregunta: {question}")
        print(f"\nRespuesta:\n{answer}")
        print("\n" + "-" * 50)

    return answer


# Inicializar la memoria
memory = ConversationMemory(max_turns=10)
print("Memoria y funcion ask() listas")

Memoria y funcion ask() listas


## Probar la interacción en el notebook

In [15]:
# Limpiar memoria antes de los ejemplos
memory.clear()

print("  Pregunta 1 ")
print("█" * 60)
_ = ask(
    "¿Cuál es el argumento principal de Pulp Fiction "
    "y quiénes son sus personajes centrales?"
)

print("  Pregunta 2 ")
print("█" * 60)
_ = ask(
    "¿Cuál es el argumento principal de Inglorious Basterds "
    "y quiénes son sus personajes centrales?"
)

print("  EJEMPLO 3 ")
print("█" * 60)
_ = ask(
    "¿Qué premios y nominaciones importantes ha recibido "
    "Quentin Tarantino a lo largo de su carrera?"
)

print("  EJEMPLO 4 ")
print("█" * 60)
_ = ask(
    "¿Como se llaman los dos agentes que interpretan Samuel L. Jackson y John Travolta?"
)

# ── Ejemplo 5: PRUEBA DE MEMORIA ───────────────────────────────
# Esta pregunta referencia explícitamente respuestas anteriores
# para demostrar que el agente mantiene el contexto entre turnos.
print("\n" + "█" * 60)
print("  EJEMPLO 5 — PRUEBA DE MEMORIA ")
print("█" * 60)
_ = ask(
    "De los premios que mencionaste antes, ¿cuál consideras "
    "el más importante para su trayectoria? "
    "¿Y cómo se relaciona con el estilo cinematográfico que describiste?"
)

Historial borrado.
  Pregunta 1 
████████████████████████████████████████████████████████████

Pregunta: ¿Cuál es el argumento principal de Pulp Fiction y quiénes son sus personajes centrales?

Respuesta:
El argumento principal de Pulp Fiction, conocida como **Tiempos violentos** en Hispanoamérica, se basa en tres historias entrelazadas que se narran en un orden no cronológico.

Los personajes centrales de la película son:

*   **Vincent Vega**, un asesino a sueldo.
*   **Butch Coolidge**, un boxeador.
*   **Jules Winnfield**, otro asesino a sueldo.

La película también cuenta con un reparto coral que incluye a:

*   **Mia Wallace**
*   **Marsellus Wallace**
*   **Pumpkin** o **Ringo**
*   **Yolanda** o **Honey Bunny**
*   **Fabienne**
*   **Winston Lobo**
*   **Lance**
*   **Jody**
*   **Capitán Koons**
*   **Paul**
*   **Trudi**
*   **Zed**
*   **The Gimp**

--------------------------------------------------
  Pregunta 2 
████████████████████████████████████████████████████████████



## Interacción con el notebook

In [ ]:
print("Chat interactivo - Experto en Tarantino")
print("Comandos: salir | limpiar | memoria")

while True:
    try:
        user_input = input("\nTu: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nSesion terminada.")
        break

    if not user_input:
        continue
    elif user_input.lower() == "salir":
        print("Hasta luego!")
        break
    elif user_input.lower() == "limpiar":
        memory.clear()
    elif user_input.lower() == "memoria":
        print(f"\nHistorial ({len(memory.history) // 2} turnos):")
        memory.show()
    else:
        ask(user_input)

Chat interactivo - Experto en Tarantino
Comandos: salir | limpiar | memoria

Tu: quien es el oso judío

Pregunta: quien es el oso judío

Respuesta:
El Oso Judío es el sargento **Donnie Donowitz**, interpretado por **Eli Roth**. Es conocido por golpear a los nazis con un bate de béisbol en la cabeza.

--------------------------------------------------

Tu: Quien es django

Pregunta: Quien es django

Respuesta:
Django es un **esclavo negro** interpretado por **Jamie Foxx**. Es liberado por el doctor King Schultz (Christoph Waltz).

--------------------------------------------------

Tu: Como acaba hitler en malditos bastardos

Pregunta: Como acaba hitler en malditos bastardos

Respuesta:
Según los fragmentos proporcionados, **Hitler muere en el cine** durante el incendio provocado por Marcel. Los Bastardos, incluyendo a Donowitz y Omar, disparan a todos los presentes, asesinando a Hitler y a Goebbels.

--------------------------------------------------

Tu: salir
Hasta luego!


## Aplicación streamlit

In [ ]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "streamlit"], check=True)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:

# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA A — Escribir app.py en disco                          ║
# ╚══════════════════════════════════════════════════════════════╝

app_code = """
import os
import streamlit as st
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Configuracion
CHROMA_DIR      = "/content/chroma_db"
COLLECTION_NAME = "tarantino_expert"
TOP_K           = 5

SYSTEM_PROMPT = (
    "Eres un asistente experto en el cine de Quentin Tarantino. "
    "Basa tus respuestas unicamente en los fragmentos recuperados. "
    "Si no tienes la informacion dilo claramente. "
    "Responde siempre en espanol."
)

st.set_page_config(page_title="Experto Tarantino", layout="wide")
st.title("Experto en Cine de Tarantino")

# Cargo el modelo y la base de datos solo una vez
@st.cache_resource
def load_agent():
    emb = GoogleGenerativeAIEmbeddings(
        model     = "gemini-embedding-001",
        task_type = "retrieval_document",
    )
    vs = Chroma(
        collection_name    = COLLECTION_NAME,
        embedding_function = emb,
        persist_directory  = CHROMA_DIR,
    )
    ret = vs.as_retriever(
        search_type   = "mmr",
        search_kwargs = {"k": TOP_K, "fetch_k": TOP_K * 3},
    )
    llm = ChatGoogleGenerativeAI(
        model       = "gemini-2.5-flash-lite",
        temperature = 0.2,
        convert_system_message_to_human = True,
    )
    return ret, llm

retriever, llm = load_agent()

# Boton para limpiar el chat en el sidebar
with st.sidebar:
    st.metric("Chunks indexados", retriever.vectorstore._collection.count())
    if st.button("Limpiar conversacion"):
        st.session_state.messages = []
        st.rerun()

# Inicializo el historial si no existe
if "messages" not in st.session_state:
    st.session_state.messages = []

# Muestro los mensajes anteriores
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# Caja de texto y boton para enviar
col1, col2 = st.columns([5, 1])
with col1:
    prompt = st.text_input(
        "Pregunta:",
        placeholder = "Pregunta sobre Tarantino...",
        label_visibility = "collapsed",
        key = "input_box",
    )
with col2:
    enviar = st.button("Enviar", use_container_width=True)

# Cuando el usuario envia una pregunta
if enviar and prompt.strip():

    st.session_state.messages.append({"role": "user", "content": prompt})

    with st.spinner("Buscando en los documentos..."):

        # Busco los fragmentos relevantes
        docs    = retriever.invoke(prompt)
        context = "\\n\\n".join([d.page_content for d in docs])

        # Construyo el historial para que el modelo tenga memoria
        history = []
        for m in st.session_state.messages[:-1]:
            if m["role"] == "user":
                history.append(HumanMessage(content=m["content"]))
            else:
                history.append(AIMessage(content=m["content"]))

        # Llamo al modelo con el contexto y la pregunta
        mensajes = [
            SystemMessage(content=SYSTEM_PROMPT),
            *history,
            HumanMessage(content=f"Contexto:\\n{context}\\n\\nPregunta: {prompt}"),
        ]
        answer = llm.invoke(mensajes).content

    st.session_state.messages.append({"role": "assistant", "content": answer})
    st.rerun()
"""

with open("/content/app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py guardado correctamente")


# ╔══════════════════════════════════════════════════════════════╗
# ║  CELDA B — Lanzar Streamlit + cloudflared                    ║
# ╚══════════════════════════════════════════════════════════════╝

import subprocess, threading, time, os, sys, re

os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

# Lanzo Streamlit en segundo plano
def run_streamlit():
    subprocess.run([
        sys.executable, "-m", "streamlit",
        "run", "/content/app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ], env=os.environ)

threading.Thread(target=run_streamlit, daemon=True).start()
print("Iniciando Streamlit...")
time.sleep(6)

# Abro el tunel para acceder desde el navegador
tunnel = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout = subprocess.PIPE,
    stderr = subprocess.STDOUT,
)

time.sleep(5)

for _ in range(30):
    line = tunnel.stdout.readline().decode("utf8", errors="ignore").strip()
    if "trycloudflare.com" in line:
        url = re.search(r'https://\S+\.trycloudflare\.com', line)
        if url:
            print(f"App disponible en: {url.group()}")
            break

app.py guardado correctamente
Iniciando Streamlit...
App disponible en: https://christine-gmt-transaction-penalties.trycloudflare.com
